### RAG with Tabalar Data and Vector Memory


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('Chatbot_rag_v2') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
!pip install langchain==0.2.17
!pip install langchain_community==0.2.19
!pip install -qU langchain-ollama==0.1.3
!pip install -qU langchain-qdrant==0.1.4

In [3]:
import os
from dotenv import load_dotenv


### Visualizar dados

In [4]:
df_silver = spark.sql("""
    Select
        c,
        cl,
        sl,
        lt0,
        lt1,
        qv
    from iceberg.silver.tbl_silver_olhovivo"""
).limit(10)

df_silver.createOrReplaceTempView("vw_tbl_silver_olhovivo")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [5]:
spark.sql("""
select * from vw_tbl_silver_olhovivo
""").toPandas()

,c,cl,sl,lt0,lt1,qv
0,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5
1,606C-10,35363,2,CANTINHO DO CÉU,CIRCULAR,2
2,2704-10,916,1,METRÔ ITAQUERA,JD. ROBRU,3
3,6042-10,33987,2,STO. AMARO,JD. TRÊS ESTRELAS,2
4,978T-10,889,1,METRÔ BARRA FUNDA,JD. GUARANI,7
5,2719-10,33700,2,METRÔ VL. MATILDE,ERMELINO MATARAZZO,7
6,1024-10,2302,1,CONEXÃO PETRÔNIO PORTELA,JD. CAROMBÉ,2
7,875P-10,34109,2,METRÔ ANA ROSA,METRÔ BARRA FUNDA,1
8,5105-10,34419,2,TERM. MERCADO,TERM. SACOMÃ,5
9,172X-10,33629,2,METRÔ TATUAPÉ,PQ. NOVO MUNDO,7


In [6]:
df_gold = spark.sql("Select * from iceberg.gold.tbl_gold_olhovivo").limit(10)

df_gold.createOrReplaceTempView("vw_tbl_gold_olhovivo")

In [7]:
spark.sql("""
select * from vw_tbl_gold_olhovivo
""").toPandas()

,Letreiro_Linha,Linha,Sentido,Destino_Linha,Origem_Linha,Quantidade_Veiculos,Prefixo_Veiculo,Veiculo_Acessivel,Horario,Latitude,Longitude
0,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,63351,True,2025-04-26 19:51:53,-23.680852,-46.636013000000005
1,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,63290,True,2025-04-26 19:52:00,-23.6811735,-46.636431
2,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,63368,True,2025-04-26 19:51:37,-23.575192,-46.640798000000004
3,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,63375,True,2025-04-26 19:52:04,-23.64941375,-46.640634750000004
4,5290-10,33365,2,TERM. PQ. D. PEDRO II,DIV. DIADEMA,5,63247,True,2025-04-26 19:51:33,-23.61655275,-46.63886600000001
5,606C-10,35363,2,CANTINHO DO CÉU,CIRCULAR,2,66111,True,2025-04-26 19:51:43,-23.742632999999998,-46.658328499999996
6,606C-10,35363,2,CANTINHO DO CÉU,CIRCULAR,2,66112,True,2025-04-26 19:51:58,-23.7406495,-46.6570705
7,2704-10,916,1,METRÔ ITAQUERA,JD. ROBRU,3,36685,True,2025-04-26 19:52:03,-23.526659000000002,-46.409887999999995
8,2704-10,916,1,METRÔ ITAQUERA,JD. ROBRU,3,36021,True,2025-04-26 19:51:44,-23.521501,-46.442832499999994
9,2704-10,916,1,METRÔ ITAQUERA,JD. ROBRU,3,36016,True,2025-04-26 19:51:57,-23.5236605,-46.418086


## Funções Auxiliares

In [10]:
import re

def clear_sql(genereted_sql):
    """Remove markdown code blocks"""
    sql = re.sub(r"```sql|```", "", genereted_sql, flags=re.IGNORECASE).strip()
    return sql

In [11]:
def get_metadata(table_name):
    df = spark.sql(f"SELECT * FROM {table_name} LIMIT 1;")
    columns = "\n".join([f"- {f.name}: {f.dataType.simpleString()}" for f in df.schema])
    return f"Tabela: {table_name}\n\nColunas:\n{columns}"


## Qdrant Memory

In [12]:
load_dotenv('../.env')
OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")


In [13]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="mistral:latest", base_url=OLLAMA_API_URL)

In [14]:
# Cria coleção para armazenar os embeddings 

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
from langchain_core.documents import Document
import uuid

client = QdrantClient(":memory:")

collection_name ="olho_vivo"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=4096, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embedding,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [15]:
%run ./Memory.ipynb

In [16]:
qdrant_memory = QdrantMemory(client, embedding)

## Iniciar Mistral 7B

In [17]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="mistral:latest", 
    base_url=OLLAMA_API_URL,
    temperature = 0.3,
 
)

# llm = ChatOllama(
#     model="mistral:latest", 
#     base_url=OLLAMA_API_URL,
#     temperature=0.3,
#     num_predict=200,
#     top_k=20,
#     repeat_penalty=1.2
    
# ) 

### Configurar Promps: Roles System e Human

In [18]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# Prompt para gerar SQL (Spark)
prompt_sql = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Você é um especialista em dados. Gere apenas a consulta SQL."),
    HumanMessagePromptTemplate.from_template(
        "Com base na estrutura da tabela abaixo:\n\n{schema}\n\n e contexto:\n\n{context}\n\n"
        "Escreva uma consulta SQL (somente a SQL sem explicação extra) para responder:\n{question}"
    )
])


# Prompt para retornar resultado ao usuario
prompt_response = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Você é um assistente de dados."),
    HumanMessagePromptTemplate.from_template(
        "Pergunta: {question}\n\nResultado da consulta:\n{result}\n\n"
        "Gere uma resposta clara e amigável para o usuário contendo apenas os resultados da consulta."
    )
])


In [19]:
# Função para gerar resposta com RAG (Tabela + Qdrant)
def augmented_response(question, table_name):
    print(f"\n💬 question: {question}")

    #Obter metadados da tabela
    schema_txt = get_metadata(table_name)

    # Buscar embeddings no Qdrant (Memoria)
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])
 

    # Gerar o SQL da query com base na pergunta
    sql_chain = prompt_sql | llm
    sql_result = sql_chain.invoke({
        "question": question,
        "schema": schema_txt,
        "context": context
    }).content.strip()


    sql_query=clear_sql(sql_result)
    print(f"\n💡 Generated SQL: {sql_query}")

    # Executar query no Spark
    try:

        result_df = spark.sql(sql_query).toPandas().to_dict(orient="records")
    except Exception as e:
        print(f"\n❌ Erro na execução da SQL: {e}")
        return

    # Gera resposta amigável para retornar ao usuario
    response_chain = prompt_response | llm
    response = response_chain.invoke({
        "question": question,
        "result": result_df
    }).content.strip()

    print(f"\n🤖 response: {response}")


    # Armazena pergunta + SQL + resultado + score na memoria Qdrant
    qdrant_memory.instruct(
        question, 
        sql_query, 
        metadata={
            "table": table_name,            
            "response": response,
            "schema": schema_txt, 
            "score": 1}
    )

    return response

In [23]:
table = "vw_tbl_gold_olhovivo"

question = "O veiculo de prefixo 63247 é acessivel?"

resp = augmented_response(question, table)


💬 question: O veiculo de prefixo 63247 é acessivel?

💡 Generated SQL: SELECT Veiculo_Acessivel FROM vw_tbl_gold_olhovivo WHERE Prefixo_Veiculo = '63247';

🤖 response: Sim, o veículo com prefixo 63247 está disponível para você.

⚠️ Embedding similar já existe. Incrementando score...

Score:  ⭐✩✩✩✩


### Listar Exemplos de Consultas

In [21]:
qdrant_memory.list_embedding_content()


🧬 Embedding 1:
question: Qual a posição atual do veiculo de prefixo 66111, me fala origem dele ?
SQL: SELECT Origem_Linha FROM vw_tbl_gold_olhovivo WHERE Prefixo_Veiculo = '66111';


In [22]:
qdrant_memory.list_scored_point("Qual a posição atual do veiculo de prefixo 66111, me fala origem dele ?")

📦 Payload: [ScoredPoint(id='328c06ca-c752-4828-b2a3-b9846d1364d3', version=0, score=0.99999999445652, payload={'page_content': "question: Qual a posição atual do veiculo de prefixo 66111, me fala origem dele ?\nSQL: SELECT Origem_Linha FROM vw_tbl_gold_olhovivo WHERE Prefixo_Veiculo = '66111';", 'table': 'vw_tbl_gold_olhovivo', 'response': 'O veículo com prefixo 66111 é de origem circular.', 'schema': 'Tabela: vw_tbl_gold_olhovivo\n\nColunas:\n- Letreiro_Linha: string\n- Linha: string\n- Sentido: string\n- Destino_Linha: string\n- Origem_Linha: string\n- Quantidade_Veiculos: int\n- Prefixo_Veiculo: string\n- Veiculo_Acessivel: boolean\n- Horario: timestamp\n- Latitude: string\n- Longitude: string', 'score': 1}, vector=None, shard_key=None, order_value=None)]


In [ ]:
spark.stop()

## "Ensinar"- ajustar comportamento manualmente

In [ ]:
sql_query =""" """

In [ ]:
spark.sql(sql_query).show()

In [ ]:
table_name = "table_name"
pergunta = ""
schema_txt = get_metadata(table_name)

qdrant_memory.instruct(
    pergunta,
    sql_query, 
    metadados=metadata={"table": table_name, "result": result_df, "schema": schema_txt, "score": 1}
)